# Student Performance Analysis


## Objective:
#Analyze student exam performance using NumPy and Pandas to identify trends and factors affecting academic success.


#Pandas and Numpy added

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
# Step 2: Load the Dataset

In [ ]:
sd = pd.read_csv("student_data.csv")

In [ ]:
sd.rename(columns={"Medu":"Mother",
          "Fedu":"Father",
          "higher":"higherEdu",
          "romantic":"romanticRs"},
          inplace = True)
print(sd.columns)

## Explore the Dataset

In [ ]:
sd.head(5)


In [ ]:
sd.tail(5)

In [ ]:
sd.shape

In [ ]:
sd.columns

In [ ]:
sd.dtypes

In [ ]:
sd.info()

In [ ]:
sd.describe()

## Clean Dataset


In [ ]:
sd.isnull()


In [ ]:
sd.duplicated(keep = "first")

In [ ]:
sd.drop_duplicates()

In [ ]:
sd.dtypes

In [ ]:
for col in sd.columns:
    print(col)
    print(sd[col].unique())
    

In [ ]:
sd["G1"].between(0,20)

In [ ]:
sd["G2"].between(0,20)

In [ ]:
sd["G3"].between(0,20)

In [ ]:
sd["age"].between(15,22)

## Phase 3: Feature Engineering


In [ ]:
sd["Total"] = sd["G1"] + sd["G2"] + sd["G3"]
print(sd)


In [ ]:
sd["Average"] = sd["Total"]/3
print(sd)


In [ ]:
def grading(avg):
    if avg > 16:
        return "Excellent"
    elif avg > 12:
        return "Good"
    elif avg > 8:
        return "Average"
    else:
        return "Poor"

sd["Performance"] = sd["Average"].apply(grading)
print(sd)

In [ ]:
def pf(g3):
    if g3 >= 10:
        return "Pass"
    elif g3 > 0:
        return "Fail"
    else:
        return "Incomplete"
    
sd["Pass/Fail"] = sd["G3"].apply(pf)
print(sd)

In [ ]:
sd["ParentEdu"] = (sd["Mother"] + sd["Father"])/2
print(sd)

In [ ]:
sd["TotalAlc"] = sd["Walc"] + sd["Dalc"]
print(sd)


## Phase 4: EDA


In [ ]:
sd[["G1", "G2", "G3", "Total", "Average", "absences", "TotalAlc", "ParentEdu"]].describe().round(2)

## Univariate Analysis — Numeric Columns


- G1→G3 shows a slight downward drift in mean score (10.91 → 10.42)
- G2 and G3 have min=0 (Incomplete cases); G1 does not (min=3)
- Absences are heavily right-skewed: mean (5.71) > median (4.0), 
  std (8.00) exceeds the mean — a few extreme outliers are driving this
- Class average (10.68) sits just above the pass threshold (10) — 
  performance overall is borderline
- TotalAlc and ParentEdu both show mid-to-low central tendency, 
  no major skew

In [ ]:
for col in ["Performance", "Pass/Fail", "sex", "school"]:
    print(sd[col].value_counts())

## Univariate Analysis — Categorical Columns

- Performance: Average is the largest group (158, ~40%), Excellent the smallest 
  (25, ~6%) — performance skews toward the lower-middle overall
- Pass/Fail: 265 passed (~67%), but 130 students (~33%) either failed (92) 
  or didn't complete finals (38) — roughly 1 in 3 students didn't pass
- Performance (94 "Poor") and Pass/Fail (130 non-passers) don't fully align, 
  since Performance uses the 3-grade average while Pass/Fail uses G3 only
- Sex is fairly balanced: 208 F vs 187 M
- School is heavily imbalanced: 349 GP vs 46 MS — any GP vs MS comparison 
  later should account for this small MS sample size

In [ ]:
for col in ["Mjob","Fjob","internet","romanticRs","higherEdu"]:
   print(sd[col].value_counts())

## Univariate Analysis — Categorical Columns (Part 2)

- Mjob and Fjob are both dominated by "other" (36% and 55% respectively), 
  limiting how useful these columns are for detailed job-based comparisons
- at_home is far more common for mothers (59) than fathers (20) — 
  reflects a real gender pattern in this sample
- internet: 83% yes vs 17% no — comparisons involving the "no" group 
  should be treated cautiously due to small sample size (66)
- romantic: 67% no vs 33% yes — reasonably usable split
- higher: 95% yes vs 5% no (only 20 students) — very low variation, 
  weak column for finding differences; "no" group too small to generalize from

In [ ]:
sd.groupby("studytime")["G3"].mean().round(2)


In [ ]:
sd["studytime"].value_counts()

## Bivariate — Studytime vs G3

- Clear upward trend from studytime 1 (10.05) to studytime 3 (11.40)
- Slight dip at studytime 4 (11.26), but this group only has 27 students 
  (vs 198 at studytime 2) — likely sample size noise, not a real reversal
- Overall conclusion: more study time is generally associated with 
  higher G3, though the relationship isn't perfectly linear

In [ ]:
sd.groupby("failures")["G3"].mean().round(2)


In [ ]:
sd["failures"].value_counts()

## Bivariate — Failures vs G3

- Strong, consistent downward trend: 11.25 → 8.12 → 6.24 → 5.69 
  as failures go from 0 to 3
- Much larger spread (~5.6 points) than studytime (~1.35 points) — 
  failures appears to be a stronger predictor of G3 than studytime
- Biggest single drop is between 0 and 1 failure (11.25 → 8.12)
- Groups 2 and 3 are small (17 and 16 students) so exact values are 
  less precise, but the consistent downward direction across all 
  groups makes the overall trend credible
- Sample sizes drop sharply: 312 students have 0 failures vs only 
  16 with 3 failures — most students in this dataset have never failed

In [ ]:
sd.groupby("sex")["G3"].mean().round(2)

In [ ]:
sd["sex"].value_counts()

In [ ]:
sd.groupby("sex")["studytime"].mean().round(2)

In [ ]:
sd.groupby("sex")["goout"].mean().round(2)

In [ ]:
sd.groupby("sex")["failures"].mean().round(2)

## Bivariate — Sex vs G3 (deeper look)

- Females show MORE favorable indicators than males: higher studytime 
  (2.28 vs 1.76), lower goout (3.03 vs 3.20), fewer failures (0.30 vs 0.37)
- Despite this, females still average LOWER G3 (9.97) than males (10.91)
- This means studytime/goout/failures do NOT explain the sex-based 
  performance gap — something else is driving it that isn't captured 
  in these three variables
- Notable, honest finding: not all patterns have a clean explanation — 
  worth flagging as a limitation/open question in final insights (Phase 7)

In [ ]:
sd.groupby("TotalAlc")["G3"].mean().round(2)

In [ ]:
sd["TotalAlc"].value_counts()

## Bivariate — TotalAlc vs G3

- No clear or consistent trend across TotalAlc levels (2 through 10) — 
  averages bounce between ~9.0 and ~10.8 without a steady direction
- Group sizes shrink drastically at higher TotalAlc values 
  (e.g., only 4-9 students at TotalAlc 8, 9, 10), making those 
  specific averages unreliable
- Conclusion: unlike failures, TotalAlc does not show a strong or 
  reliable relationship with G3 in this dataset — worth noting as 
  a "no clear effect found" result rather than forcing a pattern

In [ ]:
sd.groupby("ParentEdu")["G3"].mean().round(2)


In [ ]:
sd["ParentEdu"].value_counts()

## Bivariate — ParentEdu vs G3

- Clear upward trend from ParentEdu 1.0 (8.64) to 4.0 (11.66), 
  with a small dip at 3.5 (10.95) — overall supports "higher parental 
  education associates with higher G3"
- ParentEdu = 0.5 shows the highest average (12.50) but is based on 
  only 2 students — not statistically meaningful, excluded from the 
  overall trend interpretation
- Group sizes for 1.0 through 4.0 are all reasonably large (39-72 
  students each), making this trend reliable

In [ ]:
sd.groupby("schoolsup")["G3"].mean().round(2)

In [ ]:
sd["schoolsup"].value_counts()

## Bivariate — schoolsup vs G3

- Students without school support average HIGHER G3 (10.56, n=344) 
  than students with support (9.43, n=51)
- Important: this does NOT mean support causes lower grades — 
  schoolsup is likely given to already-struggling students, so the 
  gap reflects pre-existing difficulty, not an effect of the support
- Cannot determine causation from this comparison alone — a genuine 
  limitation to flag in Phase 7, not a finding to overstate

##Phase 5: GroupBy Analysis


In [ ]:
sd.groupby("studytime")["G3"].agg(["mean","max","min","count"]).round(2)

## GroupBy Analysis — Studytime (mean, min, max, count)

- Every studytime group has a min of 0 — Incomplete/zero-scoring 
  students exist at every study level, even studytime=4
- Max scores are nearly identical across all groups (19-20) — 
  top performers exist even at studytime=1
- The mean trend (10.05 → 11.26) still holds, but the wide range 
  within each group shows studytime shifts the average without 
  guaranteeing any individual outcome
- Confirms: studytime is associated with performance at the group 
  level, but individual variation remains high regardless of studytime

In [ ]:
sd.groupby(["studytime","sex"])["G3"].agg(["mean","count"]).round(2)

## GroupBy Analysis — Studytime x Sex vs G3 (with counts)

- Group size does NOT explain the sex gap — there's no consistent 
  relationship between which sex is the bigger group and which 
  scores lower (e.g., males are the smaller group at studytime=4 
  yet still score higher)
- The sex gap (males > females) holds at every studytime level 
  regardless of group size, confirming this is a real pattern, 
  not a sample-size artifact
- Specific values built on small groups (studytime=3 males, n=14; 
  studytime=4 males, n=10) should be read cautiously — they're 
  less stable estimates, but don't change the overall direction 
  of the gap

In [ ]:
sd.groupby(["romanticRs","sex"])["G3"].agg(["mean","count"]).round(2)

## GroupBy Analysis — Romantic x Sex vs G3

- Sex gap persists in both groups: males > females whether or not 
  they're in a relationship
- Gap widens when in a relationship: 0.56 (not in relationship) 
  vs 1.41 (in relationship)
- Key insight: being in a relationship is associated with LOWER G3 
  for both sexes, but the drop is much steeper for females (-1.54) 
  than for males (-0.69)
- All group sizes are reasonably large (53-134), so this pattern 
  is reliable, not a small-sample artifact
- This is one of the more specific findings so far: romantic status 
  doesn't just correlate with grades — it interacts differently by sex

## Phase 6: Pivot Tables


In [ ]:
pd.pivot_table(sd,
               values = "G3",
               index = "studytime",
               columns = "sex",
               aggfunc= "mean").round(2)

sex,F,M
studytime,,
1,9.85,10.12
2,9.50,11.07
3,10.73,13.86
4,11.00,11.70


In [ ]:
pd.pivot_table(sd,
               values = "G3",
               index = "school",
               columns = "sex",
               aggfunc= ["mean","count"]).round(2)

mean        count     
sex        F      M     F    M
school                        
GP      9.97  11.06   183  166
MS      9.92   9.76    25   21

## Pivot Table — School x Sex vs G3

- At GP school: males average notably higher than females (11.06 vs 9.97, 
  gap ~1.09) — matches the overall pattern found earlier
- At MS school: males and females are nearly identical (9.76 vs 9.92) — 
  the sex gap essentially disappears
- GP counts are large and reliable (183 F, 166 M); MS counts are much 
  smaller (25 F, 21 M) — the MS near-tie should be read cautiously, 
  not as strong evidence that no gap exists there
- Key refinement: the sex gap found throughout this project appears 
  concentrated at GP school, not a universal pattern across both schools

In [ ]:
pd.pivot_table(sd,
               values = "G3",
               index = "Performance",
               columns = "Pass/Fail",
               aggfunc= ["mean","count"]).round(2)

mean                   count                  
Pass/Fail    Fail Incomplete   Pass  Fail Incomplete   Pass
Performance                                                
Average      8.67        NaN  10.76  36.0        NaN  122.0
Excellent     NaN        NaN  17.92   NaN        NaN   25.0
Good          NaN        NaN  14.01   NaN        NaN  118.0
Poor         6.93        0.0    NaN  56.0       38.0    NaN

## Pivot Table — Performance x Pass/Fail

- Excellent (25) and Good (118) students always Pass — average-based 
  Performance and G3-based Pass/Fail fully agree at the top end
- Poor students (94 total) split into Fail (56) and Incomplete (38) — 
  two distinct situations: genuinely low scorers vs. disengaged/absent 
  from finals
- KEY FINDING: Average-performance students split into Fail (36) and 
  Pass (122) — this resolves the earlier mismatch (94 Poor vs 130 
  total non-passers). These 36 students had a decent overall average 
  but still failed their final exam specifically — suggesting a 
  late decline rather than year-long struggle
- Incomplete students always show mean G3 = 0.0 (by definition), 
  confirming the Pass/Fail logic is working correctly

# Phase 7: Insights and Conclusions

## 1. Dataset Overview
This dataset contains records for 395 students from two Portuguese secondary 
schools — Gabriel Pereira (GP, 349 students) and Mousinho da Silveira 
(MS, 46 students). It includes demographic, family background, lifestyle, 
and academic performance data (G1, G2, G3 grades on a 0–20 scale).

## 2. Data Quality Notes
- No major missing values or duplicate rows were found in the raw dataset.
- `absences` showed strong right-skew, with a small number of extreme 
  outliers (54, 56, 75 days) pulling the mean well above the median.
- A number of students had G3 = 0 while G1/G2 had real scores — interpreted 
  as students who did not sit the final exam, not as a true zero score. 
  These were separated into an "Incomplete" category rather than merged 
  into "Fail," preserving an important distinction in the data.

## 3. Feature Engineering Summary
The following features were created to support deeper analysis:
- **Total / Average** — combined G1+G2+G3 into a single performance score
- **Performance** — categorical label (Excellent / Good / Average / Poor) 
  based on Average
- **Pass/Fail** — Pass (G3≥10) / Fail (0<G3<10) / Incomplete (G3=0), based 
  specifically on the final grade
- **ParentEdu** — average of Medu and Fedu, representing household 
  education level
- **TotalAlc** — combined weekday + weekend alcohol consumption (Dalc+Walc)

## 4. Key Findings

### Strong, well-supported patterns
- **Past failures is the strongest predictor of G3.** Average G3 drops 
  consistently and sharply as failures increase: 11.25 (0 failures) → 8.12 
  (1) → 6.24 (2) → 5.69 (3). The trend is monotonic across all groups.
- **Study time shows a genuine, though imperfect, positive relationship 
  with G3.** Average G3 rises from 10.05 (studytime=1) to 11.40 
  (studytime=3), with a small dip at studytime=4 (11.26) likely explained 
  by that group's small size (n=27).
- **Parental education correlates positively with G3.** Average G3 rises 
  from 8.64 (ParentEdu=1.0) to 11.66 (ParentEdu=4.0), a solid upward trend 
  across well-populated groups (39–72 students each).
- **A persistent, unexplained sex gap exists in performance.** Males 
  average higher G3 than females overall (10.91 vs 9.97), and this holds 
  at every studytime level and in both romantic-status groups. Notably, 
  females report *better* study habits, *lower* goout, and *fewer* 
  failures than males — meaning this gap is not explained by any of the 
  behavioral factors examined here.
- **The sex gap is concentrated at GP school.** At GP, males outperform 
  females by ~1.09 points; at MS, the gap nearly disappears (males and 
  females are almost identical). MS's smaller sample size (46 students) 
  means this should be treated as a directional finding, not a conclusive one.
- **Romantic relationships associate with lower G3 for both sexes, but 
  far more so for females.** Females in a relationship drop 1.54 points 
  on average compared to those not in one; males drop only 0.69 points. 
  This is one of the more specific, well-supported findings in the project.

### Findings requiring careful interpretation (correlation ≠ causation)
- **Students receiving school support (schoolsup) average lower G3** 
  (9.43 vs 10.56). This does not mean support harms performance — support 
  is typically given to students already struggling, so the gap likely 
  reflects pre-existing difficulty rather than an effect of the support 
  itself.

### Weak or inconclusive patterns
- **TotalAlc (combined alcohol consumption) shows no clear or reliable 
  trend with G3.** Averages bounce between ~9.0 and ~10.8 with no 
  consistent direction, and sample sizes collapse sharply at higher 
  consumption levels (as few as 4–9 students). This is an honest 
  "no strong effect found" result rather than a confirmed relationship.

### A distinct sub-group worth noting
- Cross-referencing `Performance` and `Pass/Fail` revealed that all 
  Excellent/Good students passed, and all Poor students either failed 
  or were Incomplete — but 36 students labeled "Average" (a decent 
  overall grade average) still failed their final exam specifically. 
  This suggests a subgroup of students who performed adequately across 
  the year but declined sharply at the final exam — a different 
  situation than students who struggled consistently all year.

## 5. Limitations
- The dataset is heavily imbalanced by school (349 GP vs 46 MS) — any 
  GP vs MS comparison should be read with reduced confidence on the MS side.
- Several subgroup breakdowns (e.g., studytime=4, TotalAlc=9/10, 
  ParentEdu=0.5) are based on very few students and should not be 
  over-interpreted.
- Relationships found here (e.g., schoolsup, sex) are correlational, 
  not causal — the data cannot determine *why* these patterns exist, 
  only that they do.

## 6. Final Takeaway
Past academic failures are, by a clear margin, the strongest factor 
associated with lower final grades in this dataset. Study habits and 
parental education show meaningful but more modest positive associations 
with performance. A consistent sex-based performance gap exists that is 
not explained by studytime, goout, or failure history — favoring males, 
concentrated primarily at GP school, and widening notably for students 
in romantic relationships. Alcohol consumption, despite being a commonly 
assumed factor in student performance, showed no reliable relationship 
with grades in this particular dataset.